# Init

Create variants of force field for the NPC system

In [ ]:
import xml.etree.ElementTree as ET
import numpy as np
# https://github.com/mellofariam/mmct
from mmct.mmct.force_field import *
from copy import deepcopy
import pandas as pd
from collections import Counter
import pickle

import sys
sys.path.append('/home/ed31/Documents/LargeSystem/npc/')
from npc_ana_tool import *

In [2]:
# WARNING: ContactOnuchic() is 0 indexed while the XML file is 1 indexed
def scale_and_write_contacts(xml_tree, indices, scale_factor, newfile):
    xml_tmp = scale_contacts(xml = xml_tree, atom_pairs = indices, scale_by = scale_factor)
    save_xml(xml_tmp, newfile)

In [3]:
xml_new_d = ET.parse("../dilated/dilated_CA.xml")
xml_new_c = ET.parse("../constricted/constricted_CA.xml")

In [4]:
top_new_d = read_top("../dilated/dilated_CA.top")
top_new_c = read_top("../constricted/constricted_CA.top")

# Swap part of the force fields between constricted and dilated

In [2]:
ring_indices = get_ring_indices(df_chain_info)
ring_all_indices = get_ring_allsubunits_indices(ring_indices)

In [ ]:
IRLR_indices = ring_all_indices[1] | ring_all_indices[3]

In [ ]:
def swap_xml(
        tree1: ET.ElementTree,
        tree2: ET.ElementTree, 
        indices: set[int], 
        newfile1: str, 
        newfile2: str
): 
    '''The indices should be 0-based, while the XML files are 1-based'''
    def _condition_contact(element):
        return ((int(element.attrib['i']) - 1) in indices) and ((int(element.attrib['j']) - 1) in indices)
    
    def _condition_dihedral(element):
        return (int(element.attrib['i']) - 1) in indices
    
    def _swap(xml_1, xml_2, condition):

        pool_1 = []
        pool_2 = []
        new_xml_1 = []
        new_xml_2 = []

        # Make sure that the order of the elements in xml_1 and xml_2 is the same
        for element in xml_1:
            if element.tag != 'interaction':
                new_xml_1.append(element)
            elif condition(element):
                pool_1.append(element)
            else:
                new_xml_1.append(element)

        for element in xml_2:
            if element.tag != 'interaction':
                new_xml_2.append(element)
            elif condition(element):
                pool_2.append(element)
            else:
                new_xml_2.append(element)

        # Swap the elements
        for atom_ij in pool_1:
            new_xml_2.append(atom_ij)
        for atom_ij in pool_2:
            new_xml_1.append(atom_ij)
        
        xml_1[:] = new_xml_1
        xml_2[:] = new_xml_2

    root1 = tree1.getroot()
    root2 = tree2.getroot()

    # Contacts
    for xml_1 in root1.find('contacts'):
        for xml_tmp in root2.find('contacts'):
            if xml_1.attrib['name'] == xml_tmp.attrib['name']:
                xml_2 = xml_tmp
                break
        
        _swap(xml_1, xml_2, _condition_contact)

    # Dihedrals
    for xml_1 in root1.find('dihedrals'):
        for xml_tmp in root2.find('dihedrals'):
            if xml_1.attrib['name'] == xml_tmp.attrib['name']:
                xml_2 = xml_tmp
                break
        
        _swap(xml_1, xml_2, _condition_dihedral)
    
    # Write the new XML files
    save_xml(tree1, newfile1)
    save_xml(tree2, newfile2)

In [ ]:
def swap_top(
    top_template1: dict[str, pandas.DataFrame], 
    top_template2: dict[str, pandas.DataFrame], 
    indices: set[int], 
    newfile1: str, 
    newfile2: str
):
    '''The indices should be 0-based, while the TOP files are 1-based'''
    def _condition_1(df):
        return (df['ai'] - 1).isin(indices)
    
    def _condition_2(df):
        return (df['ai'].astype(int) - 1).isin(indices) & (df['aj'].astype(int) - 1).isin(indices)

    def _swap(df_1, df_2, condition):
        pool_1 = df_1[condition(df_1)]
        pool_2 = df_2[condition(df_2)]

        new_df_1 = pd.concat([df_1[~condition(df_1)], pool_2])
        new_df_2 = pd.concat([df_2[~condition(df_2)], pool_1])

        return new_df_1, new_df_2

    top1_out = deepcopy(top_template1)
    top2_out = deepcopy(top_template2)

    # Bonds
    bond_1 = top1_out['bonds']
    bond_2 = top2_out['bonds']
    top1_out['bonds'], top2_out['bonds'] = _swap(bond_1, bond_2, _condition_1)

    # Angles
    angle_1 = top1_out['angles']
    angle_2 = top2_out['angles']
    top1_out['angles'], top2_out['angles'] = _swap(angle_1, angle_2, _condition_1)

    # Exclusions
    exclusion_1 = top1_out['exclusions']
    exclusion_2 = top2_out['exclusions']
    top1_out['exclusions'], top2_out['exclusions'] = _swap(exclusion_1, exclusion_2, _condition_2)

    # Write the new TOP files
    save_top(top1_out, newfile1)
    save_top(top2_out, newfile2)

In [ ]:
swap_xml(
    xml_new_d, xml_new_c,
    IRLR_indices, 'dilated_IRLRconstricted.xml', 'constricted_IRLRdilated.xml'
)
! echo >> dilated_IRLRconstricted.xml
! echo >> constricted_IRLRdilated.xml

In [ ]:
swap_top(top_new_d, top_new_c, IRLR_indices, 'dilated_IRLRconstricted.top', 'constricted_IRLRdilated.top')

# Rescale unique contacts

NOTE: TOP file requires no change as no contacts are deleted

In [5]:
dire_Q = "../dual_basin/"

with open(f"{dire_Q}/contact_DB_uniq_c.pkl", "rb") as f:
    cont_uniq_c = pickle.load(f)
# with open(f"{dire_Q}/contact_DB_uniq_d.pkl", "rb") as f:
#     cont_uniq_d = pickle.load(f)

In [11]:
df_uniq_c = QFormation.get_Q_df(cont_uniq_c)

In [6]:
# indices for the unique contacts, start from 1 for XML
indices4xml_uniq_c = cont_uniq_c.getIndices() + 1

In [7]:
# Rescale by 50%
scale_and_write_contacts(xml_new_c, indices4xml_uniq_c, 0.5, 'constricted_CA_uniq50p.xml')
! echo >> constricted_CA_uniq50p.xml

In [8]:
# Rescale by 40%
scale_and_write_contacts(xml_new_c, indices4xml_uniq_c, 0.4, 'constricted_CA_uniq40p.xml')
! echo >> constricted_CA_uniq40p.xml

In [9]:
# Rescale by 30%
scale_and_write_contacts(xml_new_c, indices4xml_uniq_c, 0.3, 'constricted_CA_uniq30p.xml')
! echo >> constricted_CA_uniq30p.xml

In [10]:
# Rescale by 20%
scale_and_write_contacts(xml_new_c, indices4xml_uniq_c, 0.2, 'constricted_CA_uniq20p.xml')
! echo >> constricted_CA_uniq20p.xml

# Rescale inter-subunit IR contacts

In [5]:
dire_Q = "../dual_basin/"

with open(f"{dire_Q}/contact_DB_uniq_c.pkl", "rb") as f:
    cont_uniq_c = pickle.load(f)
# with open(f"{dire_Q}/contact_DB_uniq_d.pkl", "rb") as f:
#     cont_uniq_d = pickle.load(f)

In [ ]:
df_uniq_c = QFormation.get_Q_df(cont_uniq_c)

In [ ]:
indices4xml =\
df_uniq_c.query(
    'rings == "IR_all-IR_all" & subunit_i != subunit_j'
)[['i', 'j']].values + 1

In [24]:
scale_and_write_contacts(xml_new_c, indices4xml, 0.1, 'constricted_CA_IRinterface_10p.xml')
! echo >> constricted_CA_IRinterface_10p.xml

# Rescale linker

In [5]:
dire_Q = "../dual_basin/"

with open(f"{dire_Q}/contact_DB_uniq_c.pkl", "rb") as f:
    cont_uniq_c = pickle.load(f)

In [ ]:
df_uniq_c = QFormation.get_Q_df(cont_uniq_c)

## linker

In [26]:
indices4xml_intrachain_linker =\
df_uniq_c.query(
    'nups == "Nup35-Nup35" & subunit_i == subunit_j'
)[['i', 'j']].values + 1

indices4xml_interchain_linker =\
df_uniq_c.query(
    '(nup_i == "Nup35" | nup_j == "Nup35") & ~(nup_i == "Nup35" & nup_j == "Nup35") & subunit_i == subunit_j'
)[['i', 'j']].values + 1

In [27]:
scale_and_write_contacts(xml_new_c, indices4xml_intrachain_linker, 0.1, 'constricted_CA_intrachain_linker_10p.xml')
! echo >> constricted_CA_intrachain_linker_10p.xml

In [28]:
scale_and_write_contacts(xml_new_c, indices4xml_interchain_linker, 0.1, 'constricted_CA_interchain_linker_10p.xml')
! echo >> constricted_CA_interchain_linker_10p.xml

## IR non-linker

In [31]:
indices4xml_intrachain_irnonlinker =\
df_uniq_c.query(
    'nups != "Nup35-Nup35" & chainid_i == chainid_j & subunit_i == subunit_j & rings == "IR_all-IR_all"'
)[['i', 'j']].values + 1 

In [32]:
scale_and_write_contacts(xml_new_c, indices4xml_intrachain_irnonlinker, 0.1, 'constricted_CA_intrachain_irnonlinker_10p.xml')
! echo >> constricted_CA_intrachain_irnonlinker_10p.xml